# Used Car Price - Exploratory Data Analysis (EDA)

This notebook conducts a comprehensive Exploratory Data Analysis on the used car resale dataset. The goal is to understand the features, inspect missing values, analyze distributions, detect outliers, and discover key correlations that will guide our preprocessing and model building stages.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Data Loading and Basic Overview

In [ ]:
# Load the dataset. We use relative path from notebooks directory
csv_path = "../data/car_data.csv"
if not os.path.exists(csv_path):
    csv_path = "car data.csv"  # Fallback to root if run from root directory

df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape}")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Detailed summary of columns, data types, and non-null values
df.info()

In [ ]:
# Descriptive statistics for numerical columns
df.describe()

**Findings:**
- The dataset contains 301 records and 9 columns.
- Target variable is `Selling_Price`.
- Predictors include both numerical attributes (`Year`, `Present_Price`, `Driven_kms`, `Owner`) and categorical attributes (`Car_Name`, `Fuel_Type`, `Selling_type`, `Transmission`).
- Standardizing `Driven_kms` to `Kms_Driven` and `Selling_type` to `Seller_Type` will make downstream names cleaner.

## 2. Null Value and Completeness Analysis

In [ ]:
# Check for missing values in all columns
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

**Findings:**
- The dataset is fully complete, with zero missing values in any of the 9 columns. No major imputation strategy is required during preprocessing, but median/mode backup loaders will be created to ensure system robustness.

## 3. Target Variable Analysis: `Selling_Price`

In [ ]:
# Distribution plot of Selling Price
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(df["Selling_Price"], kde=True, ax=axes[0], color="#1a73e8")
axes[0].set_title("Selling Price Distribution")
axes[0].set_xlabel("Selling Price (Lakhs)")

sns.boxplot(x=df["Selling_Price"], ax=axes[1], color="#ea4335")
axes[1].set_title("Selling Price Boxplot")
axes[1].set_xlabel("Selling Price (Lakhs)")

plt.tight_layout()
plt.show()

**Findings:**
- The distribution of `Selling_Price` is heavily right-skewed. Most used cars sell for less than 10 Lakhs, with a few premium outliers selling up to 35 Lakhs. Skewness in the target variable suggests linear models might struggle, and ensemble tree models (Random Forest, XGBoost) may provide better predictions.

## 4. Categorical Values Counts

In [ ]:
# Value counts for major categorical attributes
cat_cols = ["Fuel_Type", "Selling_type", "Transmission", "Owner"]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, ax=axes[i], palette="Set2", hue=col, legend=False)
    axes[i].set_title(f"Value Counts - {col}")
    axes[i].set_ylabel("Count")

plt.tight_layout()
plt.show()

**Findings:**
- `Fuel_Type`: Mostly Petrol cars, followed by Diesel. CNG cars are a very small minority.
- `Selling_type`: Dealers represent the majority of sellers, with individuals accounting for a slightly smaller segment.
- `Transmission`: Manual cars dominate the dataset; Automatic models are significantly fewer.
- `Owner`: The vast majority of cars have had 0 previous owners, with very few having 1 or 3 owners.

## 5. Outliers Analysis in Numeric Predictors

In [ ]:
# Boxplots to inspect outliers in Present Price and Driven Kilometers
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x=df["Present_Price"], ax=axes[0], color="#34a853")
axes[0].set_title("Present Price Boxplot")
axes[0].set_xlabel("Present Price (Lakhs)")

sns.boxplot(x=df["Driven_kms"], ax=axes[1], color="#fbbc05")
axes[1].set_title("Driven Kms Boxplot")
axes[1].set_xlabel("Kilometers Driven")

plt.tight_layout()
plt.show()

**Findings:**
- Outliers are visible in both `Present_Price` (values > 22 Lakhs) and `Driven_kms` (values > 100,000 km). Capping outliers via the Interquartile Range (IQR) method during preprocessing will prevent large deviations from destabilizing the regression weights.

## 6. Correlation Heatmap

In [ ]:
# Standardise names temporary for heatmap
df_temp = df.rename(columns={"Driven_kms": "Kms_Driven", "Selling_type": "Seller_Type"})

# Encode categoricals to include in correlation
df_encoded = pd.get_dummies(
    df_temp.drop(columns=["Car_Name"]),
    columns=["Fuel_Type", "Seller_Type", "Transmission"],
    drop_first=False
)

# Heatmap of correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(df_encoded.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()

**Findings:**
- `Selling_Price` has a very strong positive correlation with `Present_Price` (0.88), indicating the original market price is a dominant predictor.
- `Selling_Price` is positively correlated with Diesel fuel type (0.55) and Automatic transmission (0.37).
- Strong negative correlations exist between binary dummy categories (e.g. `Seller_Type_Dealer` vs `Seller_Type_Individual` is -1.0; `Transmission_Automatic` vs `Transmission_Manual` is -1.0). One dummy variable from each binary category will be dropped to eliminate perfect multicollinearity.

## 7. Key Relationship Visualizations

In [ ]:
# 1. Age of the car vs Selling Price
import datetime
df["car_age"] = datetime.datetime.now().year - df["Year"]

sns.lmplot(data=df, x="car_age", y="Selling_Price", height=6, aspect=1.5, line_kws={"color": "red"})
plt.title("Car Age vs Selling Price")
plt.xlabel("Car Age (Years)")
plt.ylabel("Selling Price (Lakhs)")
plt.show()

In [ ]:
# 2. Present Price vs Selling Price
sns.scatterplot(data=df, x="Present_Price", y="Selling_Price", hue="Transmission", size="Driven_kms", sizes=(20, 200), alpha=0.8)
plt.title("Present Price vs Selling Price (by Transmission & Driven Kms)")
plt.xlabel("Present Price (Lakhs)")
plt.ylabel("Selling Price (Lakhs)")
plt.show()

In [ ]:
# 3. Fuel Type vs Selling Price Boxplot
sns.boxplot(data=df, x="Fuel_Type", y="Selling_Price", palette="Pastel1", hue="Fuel_Type", legend=False)
plt.title("Fuel Type vs Selling Price")
plt.xlabel("Fuel Type")
plt.ylabel("Selling Price (Lakhs)")
plt.show()

**Findings:**
- **Age vs Price**: As expected, there is a clear negative relationship. Older cars sell for significantly less.
- **Original vs Resale Price**: A strong linear relationship exists. Cars with automatic transmission and higher present price maintain a higher resale value.
- **Fuel Type vs Price**: Diesel cars command a much higher median resale price compared to Petrol and CNG vehicles, likely due to high initial purchase price and better fuel economy.

## 8. Summary of Key Insights & Next Steps

1. **Feature Engineering**: The interaction feature `price_per_km` (Present Price relative to driven kms) and `km_per_year` (average mileage per year) will help capture vehicle usage intensity.
2. **Outlier Capping**: Capping extreme values in `Present_Price` (> 22 Lakhs) and `Driven_kms` (> 96,000 km) will stabilize model weights.
3. **Collinearity Handling**: Drop redundant binary one-hot encoded features (e.g. `Transmission_Manual` and `Seller_Type_Individual`) to prevent infinite variance in OLS estimation.
4. **Modeling**: Train multiple models (Linear, Regularized, Tree ensembles) and use GridSearchCV to tune XGBoost and Random Forest, which are expected to outperform due to the skewed distributions of predictors.